# Notebook Purpose

This notebook is designed to perform an initial quality and consistency audit of the dataset before modeling.

- **Inspect dataset structure**
    - Review available files, columns, labels, and metadata.
    - Understand how images and patient-level information are organized.

- **Check for duplicate images**
    - Identify repeated image entries (exact duplicates and potential repeated paths/IDs).
    - Quantify duplicates and flag records for potential removal.

- **Analyze label consistency at patient level**
    - Find patients associated with both **positive** and **negative** labels.
    - Highlight potential annotation conflicts, temporal effects, or data leakage risks.

In [ ]:
import torch
import matplotlib.pyplot as plt
%matplotlib widget

input_path = r"C:\Users\nicco\Documents\PoliTo\Anno 2\Centrale\Tesi\LPAC_project\data\imgNum_919_imgSize_448_03-31_MC.pt"
dataset = torch.load(input_path, map_location="cpu")

In [ ]:
print("=== DATASET OVERVIEW ===")
print(f"type(dataset): {type(dataset).__name__}")
print(f"keys: {list(dataset.keys())}")
number_of_channels = dataset["images"].shape[1]
print(f"Image_size: {number_of_channels}x{dataset['image_size']}x{dataset['image_size']}")

print("\nimages number: ", len(dataset["images"]))
print("labels number: ", len(dataset["labels"]))
print("patientIDs number: ", len(dataset["patients"]))
print("Unique patientIDs: ", len(dataset["patients"].unique()))
print("number of paths: ", len(dataset["paths"]))

random_id = torch.randint(0, len(dataset["images"]), (1,)).item()
print("\n=== SAMPLE DATA ===")
print("Sample image tensor shape: ", dataset["images"][random_id].shape)
print("Sample label: ", dataset["labels"][random_id])
print("Sample patient ID: ", dataset["patients"][random_id])

print("\n=== Image example ===")
print(f"Label: {dataset['labels'][random_id]}, Patient ID: {dataset['patients'][random_id]}")
img = dataset["images"][random_id].permute(1, 2, 0) 
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis("on")



In [ ]:
# check if there patients labeled both as positive and negative
patient_ids = dataset["patients"].unique()
pathology_map = dict()

for patient in patient_ids:
    pathology_map[patient.item()] = set()
for patient, label in zip(dataset["patients"], dataset["labels"]):
    pathology_map[patient.item()].add(label.item())
    
mixed_patients = {patient: labels for patient, labels in pathology_map.items() if len(labels) > 1}
print("\n=== Patients with mixed labels ===")
if not mixed_patients:
    print("No patients with mixed labels found.")
for patient, labels in mixed_patients.items():
    print(f"Patient ID: {patient}, Labels: {labels}")


In [ ]:
# check distribution of number of photos per patient

import matplotlib.pyplot as plt
from collections import Counter

patient_ids = [p.item() if hasattr(p, "item") else p for p in dataset["patients"]]
samples_counts = list(Counter(patient_ids).values())

plt.figure(figsize=(10, 5))
plt.hist(samples_counts, bins=20, edgecolor="black")
plt.title("Distribution of number of photos per patient")
plt.xlabel("Number of images")
plt.ylabel("Number of patients")
plt.show()


In [ ]:
from pathlib import Path
import hashlib

# def find_relative_path(full_path):
#     root = Path.cwd().resolve()
#     full_path = Path(full_path).resolve()
#     return str(full_path.relative_to(root))

def get_patient_id(patient):
    return patient.item() if hasattr(patient, "item") else patient

# 1) group all images by byte hash
hash_map = {}

for patient, img, path in zip(dataset["patients"], dataset["images"], dataset["paths"]):
    patient_id = get_patient_id(patient)
    img_path = path
    img_hash = hashlib.sha256(img.detach().cpu().contiguous().numpy().tobytes()).hexdigest()
    if img_hash not in hash_map:
        hash_map[img_hash] = []

    hash_map[img_hash].append({
        "patient": patient_id,
        "path": img_path
    })

# 2) analyze each duplicate group once
same_patient_duplicates = {}       # patient_id -> list of duplicate path groups
different_patient_duplicates = []  # list of groups involving multiple patients

for img_hash, items in hash_map.items():
    if len(items) < 2:
        continue  # not a duplicate

    # split this hash group by patient
    by_patient = {}
    for info in items:
        patient_id = info["patient"]
        if patient_id not in by_patient:
            by_patient[patient_id] = []
        by_patient[patient_id].append(info["path"])

    # duplicates inside the same patient
    for patient_id, paths in by_patient.items():
        if len(paths) > 1:
            if patient_id not in same_patient_duplicates:
                same_patient_duplicates[patient_id] = []
            same_patient_duplicates[patient_id].append(paths)

    # duplicates across different patients
    if len(by_patient) > 1:
        different_patient_duplicates.append(by_patient)

print("Same-patient duplicate groups:", sum(len(v) for v in same_patient_duplicates.values()))
print("Different-patient duplicate groups:", len(different_patient_duplicates))

print("Printing some same-patient duplicates:  ")
for patient_id, groups in list(same_patient_duplicates.items())[:5]:  # Print first 5 patients
    print(f"  Patient {patient_id}:")
    for i, paths in enumerate(groups):
        paths = [path.split("\\")[-1:] for path in paths]  # Show only last 2 parts of the path
        print(f"    Group {i+1}: {paths}")
        
print("\nPrinting some different-patient duplicates:  ")
for i, group in enumerate(different_patient_duplicates[:5]):  # Print first 5 groups
    print(f"  Group {i+1}:")
    for patient_id, paths in group.items():
        paths = [path.split("\\")[-1:] for path in paths]  # Show only last 2 parts of the path
        print(f"    Patient {patient_id}: {paths}")